# M06A: Function Calling Basics

Enable AI models to interact with external tools (databases, APIs).

**Topics:**
- Function schemas and the call execution flow
- Handling tool outputs
- Weather lookup, calculator, and database query tools

---

## 🔧 Step 1: Setup

In [ ]:
import os
import json
from pathlib import Path
from dotenv import load_dotenv

import openai

load_dotenv(dotenv_path=Path("..") / ".env")

client = openai.OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
MODEL = "gpt-5-mini"


print(f"✅ Setup complete: Using {MODEL}!")

---

## 🎯 Why Function Calling?

The model has no built-in access to your data — databases, internal APIs, or business systems.

Function calling bridges this gap: you define functions the model can request, execute them yourself, and feed the results back.

**Example:**
1. **Model decides:** Call `get_weather(location='San Francisco')`
2. **You execute:** `get_weather()` → `{"temp": 22, "condition": "sunny"}`
3. **Model answers:** "It's 22°C and sunny in San Francisco!"

This pattern works for any external system — databases, APIs, payment processors, anything your code can reach.

---

## 🏗️ Part 1: Your First Function

Define a simple Python function to fetch data.

In [ ]:
# Step 1: Define a Python function
def get_weather(location):
    """Get weather for a location."""
    # Simulated weather data (temperatures in °C)
    weather_data = {
        "san francisco": {"temp": 22, "condition": "sunny"},
        "new york": {"temp": 18, "condition": "cloudy"},
        "london": {"temp": 14, "condition": "rainy"}
    }
    
    location_key = location.strip().lower()
    if location_key in weather_data:
        return weather_data[location_key]
    else:
        return {"temp": None, "condition": "unknown"}


# --------------------------------------------------------------
# Test the function
# --------------------------------------------------------------
result = get_weather("San Francisco")
print(f"Weather: {result}")

### Define the Schema

The model can't run code — but it can *decide* when a function should be called.

You describe your function as a schema and pass it via the `tools` parameter.

When the model decides to use a tool, it tells you which function to call and what arguments to pass. You execute it yourself.

In [ ]:
# Step 2: Define function schema 
# tools is a list — you can define multiple functions for the model to choose from
tools = [
    {
        "type": "function",           # Tool type (always "function" for custom tools)
        "name": "get_weather",        # Must match your Python function name
        "description": "Get the current weather for a location",  # Helps model decide when to use it
        "parameters": {               # Describes the function's arguments
            "type": "object",
            "properties": {
                "location": {
                    "type": "string",
                    "description": "The city name, e.g. San Francisco"
                }
            },
            "required": ["location"]  # Model must always provide this
        }
    }
]


# --------------------------------------------------------------
print("✅ Function schema defined")

---

## 🔄 Part 2: Complete Function Call Flow

`response.output` is a list of items that may include text or function calls — you loop through it to find any function calls.

After executing a function, you append both the call and the output back to your conversation history before making the next API call.

In [ ]:
# Complete flow using Responses API
print("🌤️  FUNCTION CALLING DEMO")
print("="*60)

user_message = "What's the weather in San Francisco?"
print(f"User: {user_message}\n")

# Step 1: Send message with tools
# conversation_items tracks the full exchange (messages, calls, outputs)
# When using a list, each item must be a dict — not a bare string
conversation_items = [{"role": "user", "content": user_message}]

response = client.responses.create(
    model=MODEL,
    input=conversation_items,
    tools=tools
)

# Step 2: Check response for function calls
# response.output is a list — may contain text, function calls, or both
tool_called = False

for item in response.output:
    if item.type == "function_call":
        tool_called = True

        # Step 3: Parse the model's decision
        call_id = item.call_id          # Unique ID linking call to output
        function_name = item.name        # Which function the model chose
        function_args = json.loads(item.arguments)  # Arguments as a dict
        
        print(f"Model decision: Call {function_name}({function_args})\n")
        
        # Step 4: Execute the function ourselves
        if function_name == "get_weather":
            result = get_weather(function_args["location"])
            print(f"Function result: {result}\n")
            
            # Step 5: Append call + output to conversation
            # The API needs both: what was called AND what it returned
            tool_dict = {
                "type": "function_call",
                "call_id": call_id,
                "name": function_name,
                "arguments": item.arguments
            }
            conversation_items.append(tool_dict)
            conversation_items.append({
                "type": "function_call_output",
                "call_id": call_id,
                "output": json.dumps(result)
            })
            
            # Step 6: Send everything back for the final answer
            final_response = client.responses.create(
                model=MODEL,
                input=conversation_items,
                tools=tools
            )
            print(f"Assistant: {final_response.output_text}")

# No function called — model answered directly
if not tool_called:
    print(f"Assistant: {response.output_text}")

# --------------------------------------------------------------

---

## 🛠️ Part 3: Helper Function

The demo above hardcodes `if function_name == "get_weather"`.  

That works for one function but doesn't scale.  

This helper uses a dictionary to map function names to functions — add a new tool by adding one line.

In [ ]:
def run_conversation(user_message, tools, available_functions):
    """Run a function-calling loop. Stops when model returns no tool calls."""
    
    # Each item must be a dict when using a list
    conversation_items = [{"role": "user", "content": user_message}]
    
    # Loop until no tool calls (max 3 rounds to prevent infinite loops)
    for _ in range(3):
        response = client.responses.create(
            model=MODEL,
            input=conversation_items,
            tools=tools
        )
        
        # Filter for function calls only
        tool_items = [
            i for i in response.output
            if i.type == "function_call"
        ]
        
        # No tool calls — return final response
        if not tool_items:
            return (response.output_text or "").strip()
        
        # Process each tool call
        for tool_item in tool_items:
            function_name = tool_item.name
            call_id = tool_item.call_id
            
            try:
                function_args = json.loads(tool_item.arguments)
            except Exception:
                return "Error: could not parse tool arguments."
            
            if function_name not in available_functions:
                return f"Error: unknown tool '{function_name}'."
            
            # Add function call to conversation
            conversation_items.append({
                "type": "function_call",
                "call_id": call_id,
                "name": function_name,
                "arguments": tool_item.arguments
            })
            
            # Execute function and add output
            result = available_functions[function_name](**function_args)
            conversation_items.append({
                "type": "function_call_output",
                "call_id": call_id,
                "output": json.dumps(result)
            })
    
    return "Error: too many tool-call rounds."


# --------------------------------------------------------------
print("✅ Helper function ready")

### Test the Helper

In [ ]:
# Map function names to actual functions
weather_functions = {
    "get_weather": get_weather
}

questions = [
    "What's the weather in New York?",
    "How's London looking today?",
    "Tell me about the weather in Tokyo"  # Not in our data — tests fallback
]

for q in questions:
    print(f"Q: {q}")
    answer = run_conversation(q, tools, weather_functions)
    print(f"A: {answer}\n")

---

## 🧠 Part 4: Calculator Function

Uses `enum` to restrict operations to valid choices (same concept from M03A's schema enforcement) and multiple numeric arguments.

In [ ]:
# Define calculator function
def calculate(operation, a, b):
    """Perform basic math operations."""
    if operation == "add":
        return {"operation": operation, "result": a + b, "error": None}
    
    if operation == "subtract":
        return {"operation": operation, "result": a - b, "error": None}
    
    if operation == "multiply":
        return {"operation": operation, "result": a * b, "error": None}
    
    if operation == "divide":
        if b == 0:
            return {"operation": operation, "result": None, "error": "Division by zero"}
        return {"operation": operation, "result": a / b, "error": None}
    
    return {"operation": operation, "result": None, "error": "Unknown operation"}

# Define schema (Responses API format)
calc_tools = [
    {
        "type": "function",
        "name": "calculate",
        "description": "Perform basic math operations (add, subtract, multiply, divide)",
        "parameters": {
            "type": "object",
            "properties": {
                "operation": {
                    "type": "string",
                    "description": "The operation to perform",
                    "enum": ["add", "subtract", "multiply", "divide"]
                },
                "a": {
                    "type": "number",
                    "description": "First number"
                },
                "b": {
                    "type": "number",
                    "description": "Second number"
                }
            },
            "required": ["operation", "a", "b"]
        }
    }
]

# --------------------------------------------------------------
print("✅ Calculator ready")

### Test the Calculator

In [ ]:
# Map function name to actual Python function
calc_functions = {
    "calculate": calculate
}

calc_questions = [
    "What is 45 plus 67?",
    "Multiply 12 by 8",
    "What's 100 divided by 4?"
]

for q in calc_questions:
    print(f"Q: {q}")
    answer = run_conversation(q, calc_tools, calc_functions)
    print(f"A: {answer}\n")

---

## 📊 Part 5: Database Query Function

Simulate a database lookup by ID.

In [ ]:
# Define database function
def get_user_info(user_id):
    """Get user information from database."""
    user_id = str(user_id)  # Handle numeric IDs from model
    
    # Simulated database
    users = {
        "123": {"name": "Alice", "email": "alice@example.com", "plan": "premium"},
        "456": {"name": "Bob", "email": "bob@example.com", "plan": "free"},
        "789": {"name": "Charlie", "email": "charlie@example.com", "plan": "business"}
    }
    
    return users.get(user_id, {"error": "User not found"})

# Define schema - Responses API format
db_tools = [
    {
        "type": "function",
        "name": "get_user_info",
        "description": "Get user information from the database by user ID",
        "parameters": {
            "type": "object",
            "properties": {
                "user_id": {
                    "type": "string",
                    "description": "The user ID to look up"
                }
            },
            "required": ["user_id"]
        }
    }
]

# --------------------------------------------------------------
print("✅ Database function ready")

### Test Database Queries

In [ ]:
# Map function name to actual Python function
db_functions = {
    "get_user_info": get_user_info
}

# Test database queries
db_questions = [
    "What is the email for user 123?",
    "Show me info for user 456",
    "What plan is user 789 on?"
]

for q in db_questions:
    print(f"Q: {q}")
    answer = run_conversation(q, db_tools, db_functions)
    print(f"A: {answer}\n")

### 💡 Key Insight

The model extracts structured data (user IDs) from natural language queries.

---

## 💪 Practice: Currency Converter

Create `convert_currency` supporting USD, EUR, GBP.

**Tasks:**
1. Define Python function
2. Define schema (use `enum` for currencies)
3. Map and test with `run_conversation`

In [ ]:
# --------------------------------------------------------------
# 💪 Exercise: Currency Converter
# --------------------------------------------------------------
# Objective: Build a currency converter with proper schema.

def convert_currency(amount, from_currency, to_currency):
    """Convert between currencies."""
    # TODO: Normalize inputs (handles "usd" vs "USD")
    # from_currency = from_currency.strip().upper()
    # to_currency = to_currency.strip().upper()
    
    # TODO: Define your exchange rates dictionary
    # rates = {"USD": 1.0, "EUR": ..., "GBP": ...}
    
    # TODO: Convert to USD first
    # usd_amount = amount / rates[from_currency]
    
    # TODO: Convert from USD to target currency
    # result = usd_amount * rates[to_currency]
    
    # TODO: Return structured result
    # return {"amount": amount, "from": from_currency, "to": to_currency, "result": round(result, 2)}
    pass

# Define schema - Responses API format (flat structure)
currency_tools = [
    # TODO: Create your tool schema
    # Remember: name, description, parameters at TOP level (not nested under "function")
    # {
    #       "type": "function",
    #       "name": "convert_currency",
    #       "description": "...",
    #       "parameters": {...}
    # }
]

currency_functions = {
    # TODO: Map function name to your function
    # "convert_currency": convert_currency
}

# Test questions
test_questions = [
    "Convert 50 EUR to GBP",
    "How much is 200 USD in euros?",
    "What's 75 GBP worth in dollars?"
]

# TODO: Uncomment to test your implementation
# for q in test_questions:
#       print(f"Q: {q}")
#       answer = run_conversation(q, currency_tools, currency_functions)
#       print(f"A: {answer}\n")

# --------------------------------------------------------------
print("💡 Build your currency converter using the TODOs as a guide!")

---

## 🎯 Key Takeaways

**🔧 Schema Format (Responses API):**
- `type`, `name`, `description`, `parameters` at top level — not nested under a `function` key

**🔄 The Flow:**
1. Send `input` list with `tools`
2. Check `response.output` for `function_call` items
3. Execute function with `item.arguments`
4. Append both `function_call` and `function_call_output` to `conversation_items`
5. Call the API again with updated `conversation_items` — get final answer via `response.output_text`

**🛠️ Scaling Pattern:**
- Dictionary maps function names → Python functions
- Add a new tool by adding one line to the dictionary

---

### 📍 Next Step

**M06B: Multiple Functions** — Define multiple functions, model selection, sequential calls.

---

## 🔧 Troubleshooting

**Schema Validation Error?**
- Check your schema structure. **Responses API** expects `name` and `parameters` at the top level of the tool object, NOT nested inside a `function` key.
- Ensure `type` is set to `"function"`.

**Model not calling function?**
- Check function description is clear
- Ensure user message relates to function capability
- Verify schema format is correct (flattened)

**"Expected an input item, but got a string"?**
- When `input` is a list, each item must be a dict
- Use `{"role": "user", "content": message}` not a bare string

**Parsing Errors (AttributeError)?**
- The Responses API returns `response.output` (a list of items).
- Do NOT use `response.choices[0].message.tool_calls`.
- Iterate through `response.output` and check `item.type`.

**JSON parsing errors?**
- Model might return malformed JSON arguments
- Add error handling with try/except around `json.loads()`
- Log raw arguments to debug

**Missing call_id error?**
- You must include `call_id` when returning the output item.
- Use `{"type": "function_call_output", "call_id": ..., "output": ...}`.

**Still having issues?**
- Copy any error message and paste it into ChatGPT, Claude, Gemini, or Grok — they're great at debugging
- Re-watch the lecture for this module
- Post to the Q&A with your error message and output

---